---
title: Overview
description: Key indicators and highlights of an unofficial bibliometric analysis of the publications of the School of Electrical and Computer Engineering, Aristotle University of Thessaloniki, based on a snapshot of Scopus data.
---

In [ ]:
import sys
from pathlib import Path

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "src" / "sece_stats").is_dir())
if str(ROOT / "src") not in sys.path:
    sys.path.insert(0, str(ROOT / "src"))

import numpy as np
import pandas as pd
from sece_stats import config, data
from sece_stats import plotting as dp

dp.apply_theme()
PARTIAL = config.PARTIAL_YEAR   # the year of the Scopus export, partial
AS_OF = config.EXPORT_DATE      # the export date (datetime.date)

In [ ]:
# Every number on this page is recomputed from data/processed with the definitions of the page it
# summarises (named in each block below), so that the overview and the detailed pages agree.
import re
import unicodedata
from functools import partial

from IPython.display import HTML
from plotly.subplots import make_subplots
from scipy.optimize import linear_sum_assignment
from sklearn.decomposition import NMF
from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
from sece_stats import metrics, periods
from sece_stats import small_groups as sg

pubs = data.load_publications()
pub_aff = data.load_publication_affiliations()
authors = data.load_publication_authors()
orgs = data.load_organizations().set_index("ror_id")
countries = data.load_countries().set_index("country_code")
meta = data.load_metadata()

HOME, AUTH = config.HOME_COUNTRY, config.AUTH_ROR
n_pubs = len(pubs)
FIRST_YEAR = int(pubs["year"].min())
LAST_FULL = periods.LAST_FULL                 # the latest full year (PARTIAL - 1)
AS_OF_TEXT = f"{AS_OF.day} {AS_OF:%B %Y}"     # the export date as text: day, month name and year
n_partial = int(pubs["year"].eq(PARTIAL).sum())
# The records of the Scopus search and those the pipeline keeps (described on Methodology & data)
n_returned = int(meta["records"]["exported"])
assert meta["records"]["kept"] == n_pubs, "text: the site's publications are the records kept"

# Five-year periods that end with the latest full year (sece_stats.periods), named by their role: the first
# one is clipped to the data, and the partial year has a label of its own outside the periods.
period_of = partial(periods.period_of, first_year=FIRST_YEAR)
P_RECENT = period_of(LAST_FULL)               # the latest five full years
P_PREV = period_of(LAST_FULL - 5)             # the five years before them
P_DECADE = period_of(LAST_FULL - 10)          # ten years before P_RECENT (Research topics, Funding & open access)
P_BASE = period_of(LAST_FULL - 20)            # twenty years before P_RECENT: the baseline of the long comparisons


def join_and(items) -> str:
    """'a', 'a and b', 'a, b and c'."""
    items = [str(i) for i in items]
    return " and ".join(items) if len(items) <= 2 else ", ".join(items[:-1]) + " and " + items[-1]


pubs["period"] = pubs["year"].map(period_of)

# ============================================ Venues & publishers (venues.ipynb): publisher groups
def fold(text: object) -> str:
    """Lower-case ASCII text without accents ('' for missing values)."""
    if not isinstance(text, str):
        return ""
    decomposed = unicodedata.normalize("NFKD", text)
    return "".join(ch for ch in decomposed if not unicodedata.combining(ch)).lower()


def title_key(text: object) -> str:
    """Matching key of a title: folded, '&' -> 'and', punctuation removed."""
    key = re.sub(r"[^0-9a-z]+", " ", fold(text).replace("&", " and "))
    return re.sub(r"\s+", " ", key).strip()


# Raw Scopus "Publisher" values -> publisher group (the current owner of the imprint);
# first matching rule wins, other publishers keep their own name.
PUBLISHER_RULES = [
    ("Wiley", r"wiley-ieee"),
    ("IEEE", r"\bieee\b|institute of electrical and electronics engineers"),
    ("IET", r"institution of engineering and technology|^iee$|institution of electrical engineers|"
            r"peter peregrinus"),
    ("Elsevier", r"elsevier|academic press|pergamon|north[- ]holland"),
    ("Springer Nature", r"springer|kluwer academic|nature research|nature publishing|biomed central|"
                        r"birkhauser"),
    ("MDPI", r"\bmdpi\b|multidisciplinary digital publishing"),
    ("Wiley", r"wiley|blackwell|hindawi"),
    ("Taylor & Francis", r"taylor (?:and|&) francis|routledge|crc press"),
    ("AIP Publishing", r"american institute of physics"),
    ("IOP Publishing", r"institute of physics|\biop\b"),
    ("ACM", r"association for computing machinery"),
    ("American Physical Society", r"american physical society"),
    ("Optica", r"optical society|\bosa\b|optica publishing"),
    ("SPIE", r"^spie$"),
    ("Frontiers", r"^frontiers"),
    ("SciTePress", r"scitepress|science and technology publications, lda"),
    ("Oxford University Press", r"oxford univ"),
    ("Cambridge University Press", r"cambridge univ"),
    ("World Scientific", r"^world scientific publishing|^world scientific$"),
    ("Emerald", r"^emerald"),
    ("IOS Press", r"^ios press"),
    ("SAGE", r"\bsage\b"),
    ("De Gruyter", r"de gruyter"),
    ("Wolters Kluwer", r"lippincott"),
    ("VDE Verlag", r"^vde verlag"),
]
PUBLISHER_RX = [(group, re.compile(rx)) for group, rx in PUBLISHER_RULES]

# DOI prefixes (the DOI registrant) -> publisher group, used when the Publisher field is empty.
DOI_PREFIX_GROUP = {
    "10.1109": "IEEE", "10.23919": "IEEE", "10.1049": "IET",
    "10.1016": "Elsevier", "10.1006": "Elsevier",
    "10.1007": "Springer Nature", "10.1023": "Springer Nature", "10.1038": "Springer Nature",
    "10.1186": "Springer Nature", "10.1140": "Springer Nature", "10.2200": "Springer Nature",
    "10.3390": "MDPI",
    "10.1002": "Wiley", "10.1111": "Wiley", "10.1155": "Wiley", "10.1100": "Wiley",
    "10.1080": "Taylor & Francis", "10.1201": "Taylor & Francis", "10.3109": "Taylor & Francis",
    "10.1179": "Taylor & Francis",
    "10.1063": "AIP Publishing", "10.1088": "IOP Publishing", "10.1145": "ACM",
    "10.1103": "American Physical Society", "10.1364": "Optica", "10.1117": "SPIE",
    "10.3389": "Frontiers", "10.5220": "SciTePress", "10.1093": "Oxford University Press",
    "10.1017": "Cambridge University Press", "10.1142": "World Scientific", "10.1108": "Emerald",
    "10.3233": "IOS Press", "10.1177": "SAGE", "10.1515": "De Gruyter", "10.2478": "De Gruyter",
    "10.1097": "Wolters Kluwer", "10.2528": "Electromagnetics Academy",
    "10.1371": "Public Library of Science", "10.1051": "EDP Sciences",
    "10.1121": "Acoustical Society of America", "10.1115": "ASME", "10.1162": "MIT Press",
    "10.1163": "Brill", "10.1556": "Akadémiai Kiadó", "10.2316": "ACTA Press", "10.4018": "IGI Global",
    "10.1504": "Inderscience", "10.3182": "IFAC", "10.2196": "JMIR Publications", "10.1055": "Thieme",
    "10.1137": "SIAM", "10.5755": "Kaunas University of Technology", "10.3144": "BME-PT",
}

# Venue names that identify the publisher (records with neither a publisher nor a DOI).
NAME_RULES = [
    ("IEEE", r"\bieee\b"),
    ("IET", r"^iee\b"),
    ("Springer Nature", r"^(?:lecture notes in|communications in computer and information science|"
                        r"ifip advances|ifmbe proceedings|mechanisms and machine science)"),
    ("CEUR-WS", r"^ceur workshop proceedings"),
    ("Optica", r"^optics infobase"),
    ("SPIE", r"^proceedings of spie"),
    ("IOS Press", r"^studies in health technology and informatics"),
    ("ACM", r"^acm international conference proceeding series"),
]
NAME_RX = [(group, re.compile(rx)) for group, rx in NAME_RULES]


def publisher_group(name: object) -> str | None:
    """Publisher group of a raw Publisher value (the raw name for small publishers)."""
    if not isinstance(name, str) or not name.strip():
        return None
    low = fold(name)
    return next((group for group, rx in PUBLISHER_RX if rx.search(low)), name.strip())


def name_group(*names: object) -> str | None:
    """Publisher implied by a source title or conference name, if any."""
    for name in names:
        low = fold(name)
        for group, rx in NAME_RX:
            if low and rx.search(low):
                return group
    return None


group = pubs["publisher"].map(publisher_group).astype(object)
# IET journals that Scopus lists under Wiley, the IET's publishing partner since 2021
group[pubs["source_title"].str.match(r"^IET\b", na=False) & group.eq("Wiley")] = "IET"
# no Publisher value: the DOI prefix, then a source or conference name that names the publisher
doi_prefix = pubs["doi"].str.extract(r"^\s*(10\.\d{4,9})/", expand=False)
by_doi = doi_prefix.map(DOI_PREFIX_GROUP)
take = group.isna() & by_doi.notna()
group[take] = by_doi[take]
by_name = pd.Series([name_group(s, c) for s, c in zip(pubs["source_title"], pubs["conference_name"])],
                    index=pubs.index, dtype=object)
take = group.isna() & by_name.notna()
group[take] = by_name[take]
is_journal = pubs["source_kind"].eq("journal")
tkey = pubs["source_title"].map(title_key)
# Titles that share an ISSN are one journal (renamed journals, spelling variants).
_parent: dict[str, str] = {}


def _root(key: str) -> str:
    while _parent.setdefault(key, key) != key:
        key = _parent[key]
    return key


for _, titles in pubs.loc[is_journal & pubs["issn"].notna()].groupby("issn")["source_title"]:
    keys = sorted(set(titles.map(title_key)))
    for key in keys[1:]:
        _parent[_root(key)] = _root(keys[0])
jkey = tkey.map(_root)
# last fallback for journals: the publisher of the journal's other records, when they agree
known = pd.DataFrame({"journal": jkey, "group": group})[is_journal & group.notna()]
journal_group = known.groupby("journal")["group"].agg(lambda s: s.iloc[0] if s.nunique() == 1 else None)
by_journal = jkey.map(journal_group)
take = group.isna() & is_journal & by_journal.notna()
group[take] = by_journal[take]
pubs["publisher_group"] = group.fillna("Unknown")

full = pubs[pubs["year"] < PARTIAL]           # full years only: shares, trends and comparisons
by_period = full.groupby("period")
recent = full[full["period"].eq(P_RECENT)]

n_ieee = int(pubs["publisher_group"].eq("IEEE").sum())
share_ieee = n_ieee / n_pubs
publisher_share = pd.crosstab(full["period"], full["publisher_group"], normalize="index")
ieee_recent = publisher_share.at[P_RECENT, "IEEE"]
mdpi_prev, mdpi_recent = publisher_share.at[P_PREV, "MDPI"], publisher_share.at[P_RECENT, "MDPI"]

# ============================================ Research output (output.ipynb): full counts
per_year = pubs.groupby("year").size().reindex(range(FIRST_YEAR, PARTIAL + 1), fill_value=0)
best_year = int(per_year.loc[:LAST_FULL].idxmax())
best_n, last_n = int(per_year[best_year]), int(per_year[LAST_FULL])
# the latest full year, and the record year when it is another one
last_year_text = (f"a record {dp.fmt_int(last_n)} papers in {LAST_FULL}, the latest full year"
                  if best_year == LAST_FULL else
                  f"{dp.fmt_int(last_n)} papers in {LAST_FULL}, the latest full year (the record is "
                  f"{dp.fmt_int(best_n)}, in {best_year})")
period_n = by_period.size()
recent_n, prev_n = int(period_n[P_RECENT]), int(period_n[P_PREV])
growth_recent = recent_n / prev_n - 1
growth_text = f"{dp.fmt_pct(abs(growth_recent))} {'more' if growth_recent >= 0 else 'fewer'}"
team = by_period["n_authors"].mean()          # mean authors per publication

# ============================================ Citation impact (impact.ipynb)
cites = pubs["cited_by"]                       # Scopus "Cited by" at the export date
total_cites = int(cites.sum())
h_all = metrics.h_index(cites)
TEN_FROM = periods.bounds(P_PREV)[0]           # the latest ten full years are TEN_FROM..LAST_FULL
h_ten = metrics.h_index(pubs.loc[pubs["year"].between(TEN_FROM, LAST_FULL), "cited_by"])
TOP_FRAC = 0.10
top10_share = metrics.top_share(cites, frac=TOP_FRAC)   # share of all citations held by the most-cited 10%
# Normalised citation score: citations / mean citations of the School's papers of the same year and
# type (reviews with articles: most years have too few reviews). Articles, reviews and conference papers of
# NCS_FROM..NCS_TO, as on Citation impact: papers with at least two full calendar years of citations, from the
# first year after the last one with fewer than NCS_MIN articles (reviews included) or conference papers.
CITABLE = ["Article", "Review", "Conference paper"]
NCS_MIN = 10
NCS_TO = LAST_FULL - 2
citable = pubs[pubs["doc_type"].isin(CITABLE)]
type_year = (pd.crosstab(citable["year"], citable["doc_type"].replace({"Review": "Article"}))
             .reindex(index=range(FIRST_YEAR, NCS_TO + 1), columns=["Article", "Conference paper"], fill_value=0))
thin = type_year.min(axis=1) < NCS_MIN        # a year with fewer than NCS_MIN papers of a type
NCS_FROM = int(thin[thin].index.max()) + 1 if thin.any() else FIRST_YEAR
# the same concentration within each publication year (papers of one year have had the same time)
cohort_top10 = pd.Series({y: metrics.top_share(cites[pubs["year"] == y], frac=TOP_FRAC)
                          for y in range(NCS_FROM, NCS_TO + 1)}).median()
ncs = pubs[pubs["year"].between(NCS_FROM, NCS_TO) & pubs["doc_type"].isin(CITABLE)].copy()
ncs["ncs_type"] = ncs["doc_type"].replace({"Review": "Article"})
ncs["score"] = metrics.normalized_citation_score(ncs, by=["year", "ncs_type"])
is_intl = ncs["collab_type"].eq("international")
sece_led = ncs["sece_first_author"] | ncs["sece_last_author"]          # an SECE first or last author
ncs_intl = ncs.loc[is_intl, "score"].mean()
ncs_intl_led = ncs.loc[is_intl & sece_led, "score"].mean()
ncs_intl_partner = ncs.loc[is_intl & ~sece_led, "score"].mean()       # SECE holds neither position
ncs_funded = ncs.loc[ncs["has_funding"], "score"].mean()
# "mainly those led by partners": the partner-led papers' share of the international papers' score above 1
intl_score = ncs.loc[is_intl, "score"]
partner_excess_share = (intl_score - 1)[~sece_led[is_intl]].sum() / (intl_score - 1).sum()


def vs_average(score: float) -> str:
    """Mean score relative to the School average (1): '20% above'."""
    return f"{dp.fmt_pct(abs(score - 1), 0)} {'above' if score >= 1 else 'below'}"


# ============================================ Research topics (topics.ipynb): topic model of the titles
# Research-generic words: dropped everywhere.
GENERIC = set("""
using use used uses based approach approaches analysis analyses novel new study studies method
methods methodology methodologies application applications applied toward via technique techniques
case paper proposed propose improved improving improve efficient efficiently effective effect
evaluation evaluating investigation investigating performance framework problem result issue
aspect overview review survey introduction comparison comparative comparing assessment development
developing implementation implementing editorial guest special preface reply author discussion part
erratum corrigendum multi non co self semi sub inter intra ultra anti pre post re bi tri de vs et al
generalized simple simplified considering consideration role impact way type various art prospect
challenge trend perspective opportunity insight lesson tutorial
""".split())
# scikit-learn's English stop list also holds words that are terms in engineering titles.
UNSTOP = set("""back bill bottom call describe detail empty fill fire front full interest mill move
serious side system thick thin third top""".split())
UNITS = set("kv ghz mhz khz thz hz gbps mbps kbps db dbm nm km cm mm kw mw gw kwh mwh".split())
STOP = (set(ENGLISH_STOP_WORDS) - UNSTOP) | GENERIC | UNITS
# Generic modifiers: not a term on their own, only inside two-word terms (real time, power system).
WEAK = set("""
model modeling modelling design time high higher low lower order multiple dynamic rate scheme
structure finite domain term level mode state line field current process optimal integrated active
large small scale short long real single dual parameter function condition characteristic property
network algorithm system data power future recent advance advanced emerging
back bill bottom call describe detail empty fill front full interest mill move side thick thin third top
""".split())
# Phrases folded into one token, so that spelled-out and abbreviated forms count together.
PHRASES = [(re.compile(rx), f" {tok} ") for rx, tok in [
    (r"finite[- ]difference[- ]time[- ]domain", "fdtd"),
    (r"internet[- ]of[- ]things", "iot"),
    (r"non[- ]?orthogonal[- ]multiple[- ]access", "noma"),
    (r"multiple[- ]input[- ]multiple[- ]output", "mimo"),
    (r"reconfigurable[- ]intelligent[- ]surfaces?", "ris"),
    (r"unmanned[- ]aerial[- ]vehicles?", "uav"),
    (r"orthogonal[- ]frequency[- ]division[- ]multiplexing", "ofdm"),
    (r"electroencephalogra(?:m|ms|phy|phic)", "eeg"),
    (r"covid[- ]?19|sars[- ]cov[- ]2|coronavirus", "covid"),
    (r"field[- ]programmable[- ]gate[- ]arrays?", "fpga"),
    (r"amplify[- ]and[- ]forward", "af"),
    (r"decode[- ]and[- ]forward", "df"),
    (r"millimeter[- ]wave|\bmm[- ]?waves?\b", "mmwave"),
    (r"\bpvs?\b", "photovoltaic"),
    (r"\bevs?\b", "electric vehicle"),
    (r"\bai\b", "artificial intelligence"),
]]
KEEP_S = set("""bias gas canvas atlas alias lens chaos cmos mos dos qos gps mems hems ems bms dms
series species news always thus versus plus status corpus focus stimulus campus virus consensus
apparatus nexus modulus radius locus bus mass class less loss process access stress glass cross pass
wireless lossless seamless ris res""".split())
IRREGULAR = {"indices": "index", "matrices": "matrix", "vertices": "vertex", "analyses": "analysis",
             "criteria": "criterion", "phenomena": "phenomenon"}
DASHES = dict.fromkeys(map(ord, "‐‑‒–—―−"), "-")
BRACKETS = re.compile(r"\s*;?\s*\[[^\]]*\]")   # translated titles "; [...]", "[letter]"
WORD = re.compile(r"[^\W_]+(?:'[^\W_]+)?")


def normalize_title(title: str) -> str:
    """NFKC, plain hyphens, no bracketed translations, lower case."""
    text = unicodedata.normalize("NFKC", title).translate(DASHES)
    return BRACKETS.sub(" ", text).replace("’", "'").lower()


def singular(word: str) -> str:
    """Conservative English singular (networks -> network, technologies -> technology)."""
    if word in IRREGULAR:
        return IRREGULAR[word]
    if len(word) <= 3 or word in KEEP_S or not word.endswith("s") or word.endswith(("ss", "us", "is", "ics")):
        return word
    if word.endswith("ies") and len(word) > 4:
        return word[:-3] + "y"
    if word.endswith(("sses", "ches", "shes", "xes")):
        return word[:-2]
    return word[:-1]


def tokenize(title: str) -> tuple[list[str], list[str]]:
    """(normalised tokens, surface forms) of a title; hyphenated words are split."""
    text = normalize_title(title)
    for rx, tok in PHRASES:
        text = rx.sub(tok, text)
    raw = [m.group(0).replace("'s", "").replace("'", "") for m in WORD.finditer(text)]
    return [singular(w) for w in raw], raw


def is_content(word: str) -> bool:
    return len(word) > 1 and word not in STOP and not re.fullmatch(r"\d+(?:\.\d+)?", word)


def title_terms(title: str) -> set[str]:
    """Unigrams (content words that are not generic modifiers) and bigrams of adjacent content words."""
    toks, _ = tokenize(title)
    keep = [is_content(w) for w in toks]
    uni = {w for w, k in zip(toks, keep) if k and w not in WEAK}
    bi = {f"{a} {b}" for a, ka, b, kb in zip(toks, keep, toks[1:], keep[1:]) if ka and kb and a != b}
    return uni | bi


K_TOPICS = 12                                  # the number of topics of Research topics (highest coherence)
corpus = pubs[pubs["doc_type"].ne("Erratum")].reset_index(drop=True)   # errata repeat corrected titles
corpus["terms"] = [sorted(title_terms(t)) for t in corpus["title"]]
tfidf = TfidfVectorizer(analyzer=lambda terms: terms, min_df=5, sublinear_tf=True)
X = tfidf.fit_transform(corpus["terms"])
vocab = tfidf.get_feature_names_out()
nmf = NMF(n_components=K_TOPICS, init="nndsvda", random_state=0, max_iter=2000).fit(X)
W, H = nmf.transform(X), nmf.components_
TOPICS = {   # label: (broad area, anchor terms)
    "Electricity markets": (0, ["market", "electricity market", "day ahead", "clearing"]),
    "Distribution grids & renewables": (0, ["distribution network", "active distribution", "ancillary service", "photovoltaic"]),
    "Energy management & storage": (0, ["energy storage", "storage", "battery", "building", "zero energy"]),
    "Lightning & high voltage": (0, ["lightning", "overhead", "transmission line", "impulse"]),
    "Wireless & optical communications": (1, ["wireless", "wireless communication", "optical wireless", "sensor network"]),
    "Fading channels & diversity": (1, ["fading", "fading channel", "diversity", "nakagami"]),
    "Antennas & metamaterials": (1, ["antenna", "antenna array", "metamaterial", "array"]),
    "Computational EM & photonics": (1, ["fdtd", "wave", "propagation", "waveguide", "graphene"]),
    "Machine learning": (2, ["learning", "machine learning", "deep learning", "federated learning"]),
    "Neural networks & forecasting": (2, ["neural network", "neural", "recurrent", "forecasting"]),
    "Image & signal analysis": (2, ["image", "segmentation", "detection", "classification", "fuzzy"]),
    "Control & robotics": (2, ["control", "robot", "prescribed", "tracking"]),
}
# The labels are matched to the model's topics through their anchor terms (optimal assignment).
labels = list(TOPICS)
pos = {t: i for i, t in enumerate(vocab)}
Hn = H / H.max(axis=1, keepdims=True)
anchor_score = np.array([[sum(Hn[c, pos[a]] for a in anchors if a in pos) for c in range(K_TOPICS)]
                         for _, anchors in TOPICS.values()])
lab_idx, comp_idx = linear_sum_assignment(-anchor_score)
comp_label = {c: labels[li] for li, c in zip(lab_idx, comp_idx)
              if len(set(vocab[np.argsort(H[c])[::-1][:10]]) & set(TOPICS[labels[li]][1])) >= 2}
assert len(comp_label) == K_TOPICS, "every topic of the model carries its label, as on Research topics"
has_terms = W.max(axis=1) > 0
corpus["topic"] = np.where(has_terms, [comp_label[c] for c in W.argmax(axis=1)], "Unassigned")
titles_full = corpus[corpus["year"] < PARTIAL]
topic_share = pd.crosstab(titles_full["topic"], titles_full["year"].map(period_of), normalize="columns")
topic_change = (topic_share[P_RECENT] - topic_share[P_DECADE]).drop("Unassigned", errors="ignore").sort_values()
topic_gain, topic_losses = topic_change.index[-1], list(topic_change.index[:3])
gain_from, gain_to = topic_share.at[topic_gain, P_DECADE], topic_share.at[topic_gain, P_RECENT]
losses_text = join_and(f"{t} ({dp.fmt_pct(topic_share.at[t, P_DECADE])} to {dp.fmt_pct(topic_share.at[t, P_RECENT])})"
                       for t in topic_losses)

# ============================================ Collaboration (collaboration.ipynb)
collab_share = pd.crosstab(full["period"], full["collab_type"], normalize="index")
intl, single = collab_share["international"], collab_share["single-institution"]
intl_year = full["collab_type"].eq("international").groupby(full["year"]).mean()
# External-partner values count only affiliations of co-authors who are not SECE authors (the
# *_external fields): an SECE author's own second affiliation abroad no longer makes a paper international.
intl_ext = full["is_international_external"].groupby(full["period"]).mean()
intl_year_ext = full["is_international_external"].groupby(full["year"]).mean()
n_intl_recent = int(recent["is_international"].sum())
n_own_intl_recent = int((recent["is_international"] & ~recent["is_international_external"]).sum())

# Small groups. Next to the address-based value, an external-partner value gives the number of papers that are
# international only through SECE authors' own second affiliations abroad, and values on both bases can be combined,
# on this page and with those of other pages (a period minus its years, all years minus a period, ...). None of these
# numbers may be shown, or follow from the values shown, unless it is zero or at least config.MIN_SECE_HOLDERS
# distinct SECE authors hold these affiliations: with fewer it would in practice be one or two people's paper count
# (no per-person metrics). The site's rule (sece_stats.small_groups, the same for every page) decides which values
# the pages give on both bases and checks them all together, with what a reader knows of the values left out (at
# least one paper, at most the international papers). This page gives the external-partner share only in the years
# where Collaboration splits its international bars, and for periods and years that the rule publishes (checked at
# the end of this cell).
MIN_HOLDERS = config.MIN_SECE_HOLDERS
usable = pub_aff[pub_aff["match_method"].ne("ignored")]    # society memberships are not affiliations
rule = sg.site()
CHART_FROM = 1995   # first year of the chart (earlier years have few records, as in Collaboration's annual chart)
CHART_YEARS = range(CHART_FROM, LAST_FULL + 1)
assert rule.chart_years == list(CHART_YEARS), "the chart has the years of the site's rule (Collaboration's chart)"
ext_year_shown = rule.split_shown               # the years whose external-partner share the chart shows


def fold_text(ratio: float) -> str:
    """A growth by a factor of about two to four in words, as on Collaboration: 'doubled', 'more than doubled',
    'nearly tripled'..."""
    words = {2: "doubled", 3: "tripled", 4: "quadrupled"}
    assert 1.9 <= ratio < 4.1, f"text: no wording for a growth by {ratio:.2f} times"
    k = round(ratio)
    if abs(ratio - k) <= 0.1:
        return words[k]
    if ratio < k <= ratio + 0.25:
        return f"nearly {words[k]}"
    return f"more than {words[int(ratio)]}"


intl_fold = fold_text(intl[P_RECENT] / intl[P_BASE])
# The latest full year against the earlier full years, as on Collaboration: the highest share, or after a high. As
# there, the year of the highest external-partner share is named with its share, so the chart must show it (small
# groups: checked at the end of this cell).
best_intl_year, best_ext_year = int(intl_year.idxmax()), int(intl_year_ext.idxmax())
intl_last_text = ("the highest share of any full year" if best_intl_year == LAST_FULL else
                  f"after a high of {dp.fmt_pct(intl_year[best_intl_year])} in {best_intl_year}")
if best_ext_year == LAST_FULL:
    ext_last_text = "also the highest" if best_intl_year == LAST_FULL else "the highest of any full year"
else:
    ext_last_text = f"below the {dp.fmt_pct(intl_year_ext[best_ext_year])} of {best_ext_year}"
# Partner institutions: top-level ROR organisations of the affiliations outside AUTH (usable: above).
partners = pd.Index(usable.loc[~usable["is_auth"] & usable["ror_id_top"].notna()
                               & usable["ror_id_top"].ne(AUTH), "ror_id_top"].unique())
partner_country = orgs["country_code"].reindex(partners)
n_partners = len(partners)
n_partners_gr = int(partner_country.eq(HOME).sum())
# Abroad as on Collaboration and Global reach: a foreign institution named in an affiliation located
# outside Greece; a foreign one seen only through a campus or branch in Greece is counted apart.
located_abroad = usable["country_code"].notna() & usable["country_code"].ne(HOME) & ~usable["is_auth"]
partners_abroad = set(usable.loc[located_abroad, "ror_id_top"]) & set(
    partner_country.index[partner_country.notna() & partner_country.ne(HOME)])
n_partners_abroad = len(partners_abroad)
n_partners_branch = int((partner_country.notna() & partner_country.ne(HOME)).sum()) - n_partners_abroad
partners_note = (f"since {FIRST_YEAR}: {dp.fmt_int(n_partners_gr)} in Greece, {dp.fmt_int(n_partners_abroad)} abroad"
                 + (f", {dp.fmt_int(n_partners_branch)} foreign only via a Greek campus" if n_partners_branch else ""))

# ============================================ Global reach (geography.ipynb)
# One row per (paper, country abroad); AUTH affiliations count as Greece.
geo = usable.assign(country=usable["country_code"].where(~usable["is_auth"], HOME))
links = (geo.loc[geo["country"].notna() & geo["country"].ne(HOME), ["eid", "year", "country"]]
         .drop_duplicates(["eid", "country"]))
assert set(links["eid"]) == set(pubs.loc[pubs["is_international"], "eid"]), "country rule out of sync"
# partner institutions abroad with the Global reach code: the same set as above (and on Collaboration)
inst_abroad = set(geo.loc[geo["country"].notna() & geo["country"].ne(HOME) & ~geo["is_auth"]
                          & geo["ror_id_top"].notna() & geo["ror_id_top"].map(orgs["country_code"]).ne(HOME),
                          "ror_id_top"])
assert inst_abroad == partners_abroad, "partner institutions abroad differ between the pages"
n_intl = int(pubs["is_international"].sum())
share_intl_all = n_intl / n_pubs
by_country = (links.groupby("country")["eid"].nunique().rename("papers").to_frame()
              .join(countries[["country_name", "continent"]])
              .sort_values(["papers", "country_name"], ascending=[False, True]))
n_countries = len(by_country)
n_continents = by_country["continent"].nunique()
countries_year = (links.groupby("year")["country"].nunique()
                  .reindex(range(int(links["year"].min()), PARTIAL + 1), fill_value=0))
countries_early = countries_year.loc[slice(*periods.bounds(P_BASE))].mean()     # partner countries a year
countries_recent = countries_year.loc[slice(*periods.bounds(P_RECENT))].mean()


def country_prose(code: str) -> str:
    """Country name for running text ('the United Kingdom')."""
    name = countries.at[code, "country_name"]
    return f"the {name}" if name.startswith("United ") or name in {"Netherlands", "Philippines"} else name


(country_1, country_1_n), (country_2, country_2_n) = by_country["papers"].iloc[:2].items()
country_1, country_2 = country_prose(country_1), country_prose(country_2)

# ============================================ Funding & open access (funding.ipynb)
funded = by_period["has_funding"].mean()      # acknowledges at least one funder
oa = by_period["oa_any"].mean()               # open access in any form (Scopus flag at the export)
oa_kind = recent.groupby("source_kind")["oa_any"].mean()

# ============================================ Research community (people.ipynb): aggregates only
sece = authors[authors["is_sece"]]            # one row per (paper, SECE-affiliated author)
active_year = sece[["author_id", "year"]].drop_duplicates().groupby("year").size()   # profiles a year
papers_year = sece.groupby("year")["eid"].nunique()                                    # papers with one
base_years, recent_years = slice(*periods.bounds(P_BASE)), slice(*periods.bounds(P_RECENT))
active_early = active_year.loc[base_years].sum() / periods.WIDTH       # profiles a year, on average
active_recent = active_year.loc[recent_years].sum() / periods.WIDTH
active_growth = active_recent / active_early
papers_growth = papers_year.loc[recent_years].sum() / papers_year.loc[base_years].sum()

# ============================================ Methodology & data (metadata.json)
matching = meta["matching"]["all"]            # affiliation occurrences, society memberships aside
coverage = matching["occurrences_matched"] / matching["occurrences"]
matching_ext = meta["matching"]["excluding_auth"]   # the occurrences outside AUTH: partner counts rest on these
coverage_ext = matching_ext["occurrences_matched"] / matching_ext["occurrences"]
ror_date = pd.Timestamp(meta["ror_date"])
ROR_TEXT = f"{meta['ror_version']} of {ror_date.day} {ror_date:%B %Y}"

# The text states these patterns of the current data: stop if new data change them.
assert team[P_RECENT] > team[P_BASE], "text: teams grew"
assert fold_text(intl_ext[P_RECENT] / intl_ext[P_BASE]) == intl_fold, \
    "text: the international share grew by the same word on the external-partner basis"
assert single[P_RECENT] < single[P_BASE], "text: single-institution papers fell"
assert ncs_intl > 1 and ncs_funded > 1, "text: both groups are cited above the average"
assert ncs_intl_partner > ncs_intl > ncs_intl_led, "text: the international advantage sits in partner-led papers"
assert partner_excess_share > 0.5, "text: mainly those led by partners (most of the international papers' excess)"
assert top10_share > 0.5, "text: citations are concentrated (the most-cited 10% of the papers hold most of them)"
assert cohort_top10 < top10_share, "text: concentration is lower within publication years"
assert mdpi_recent > mdpi_prev, "text: MDPI grew"
assert topic_change.iloc[-1] > 0 > topic_change.iloc[2], "text: one topic gained most, three lost most"
assert country_1_n > country_2_n, "text: the most frequent partner country is ahead of the second"
assert countries_recent > countries_early, "text: more partner countries a year"
assert funded[P_RECENT] > funded[P_DECADE] and oa[P_RECENT] > oa[P_DECADE], "text: funding and OA rose"
assert oa_kind["journal"] > 2 * oa_kind["conference"], "text: journals open far more often"
assert 1 < active_growth < papers_growth, "text: more profiles, and papers with an SECE author grew more"

# Small groups, checked: every value that the page gives on both bases (the dashed line of the chart, P_BASE and
# P_RECENT in the KPI and the highlights, LAST_FULL and the year of the highest external-partner share in the text)
# is one that the site's rule publishes, so that its audit of the values of all pages together covers this page.
page_cells = ([[y] for y in CHART_YEARS if ext_year_shown[y]]                              # the dashed line
              + [sg.years_of(P_BASE), sg.years_of(P_RECENT), [LAST_FULL], [best_ext_year]])   # KPI, highlights, text
small_groups = {
    "the page gives no value on both bases that the site's rule does not publish":
        all(rule.is_published(cell) for cell in page_cells),
    "the papers the text counts as international only through own affiliations abroad are the rule's":
        n_own_intl_recent == rule.value(sg.years_of(P_RECENT)),
    "the site's rule: its own checks, and its audit of the values of all pages together": all(rule.checks().values()),
}
assert all(small_groups.values()), f"small groups: {[k for k, ok in small_groups.items() if not ok]}"


SECE Research Analytics is an unofficial bibliometric analysis of the publications of the
School of Electrical and Computer Engineering (SECE) of the Aristotle University of Thessaloniki
(AUTH), Greece. It uses a single snapshot of Scopus data: the {eval}`dp.fmt_int(n_pubs)` records, published
from {eval}`str(FIRST_YEAR)` to {eval}`str(PARTIAL)`, that it selects from the {eval}`dp.fmt_int(n_returned)`
returned by an affiliation search for the School on {eval}`AS_OF_TEXT` (see
[Methodology & data](methodology.ipynb)), with their citation counts on that date. {eval}`str(PARTIAL)` is a
partial year ({eval}`dp.fmt_int(n_partial)` records up to the export date): it is included in the totals and
in the yearly counts, marked as partial, but left out of shares by year, trends, growth rates and
comparisons between periods. The periods are five-year spans counted back from the latest full year,
{eval}`str(LAST_FULL)`: {eval}`P_RECENT`, {eval}`P_PREV` and so on. This page summarises the main
indicators and links to the detailed pages.

In [ ]:
dp.kpi_html([
    dp.KPI("Publications", n_pubs, f"{FIRST_YEAR}–{PARTIAL} ({PARTIAL} to {AS_OF.day} {AS_OF:%B})"),
    dp.KPI("Citations", total_cites, f"Scopus “Cited by” on {AS_OF_TEXT}"),
    dp.KPI("h-index", h_all, f"all years; {h_ten} for the papers of {TEN_FROM}–{LAST_FULL}"),
    dp.KPI("International co-authorship", dp.fmt_pct(intl[P_RECENT]),
           f"papers of {P_RECENT} with an affiliation abroad; {dp.fmt_pct(intl_ext[P_RECENT])} counting only "
           "affiliations of co-authors who are not SECE authors"),
    dp.KPI("Partner institutions", n_partners, partners_note),
    dp.KPI("Partner countries", n_countries, f"on {n_continents} continents, since {FIRST_YEAR}"),
    dp.KPI("Open access", dp.fmt_pct(oa[P_RECENT]), f"publications of {P_RECENT}"),
    dp.KPI("Funding acknowledged", dp.fmt_pct(funded[P_RECENT]), f"publications of {P_RECENT}"),
], min_width=140)

In [ ]:
#| label: cell-overview-output
INTL_COLOR = dp.CATEGORY_COLORS["collab_type"]["international"]
intl_n = pubs["collab_type"].eq("international").groupby(pubs["year"]).sum()
intl_ext_n = pubs["is_international_external"].groupby(pubs["year"]).sum()
yearly = pd.DataFrame({"papers": per_year, "intl": intl_n.reindex(per_year.index, fill_value=0),
                       "intl_ext": intl_ext_n.reindex(per_year.index, fill_value=0)})
yearly["greece"] = yearly["papers"] - yearly["intl"]
yearly["share"] = yearly["intl"] / yearly["papers"]
yearly["share_ext"] = yearly["intl_ext"] / yearly["papers"]   # the external-partner share
n_before_chart = int(yearly.loc[:CHART_FROM - 1, "papers"].sum())
max_before_chart = int(yearly.loc[:CHART_FROM - 1, "papers"].max())
yearly = yearly.loc[CHART_FROM:]

fig = make_subplots(rows=2, cols=1, shared_xaxes=True, vertical_spacing=0.08, row_heights=[0.62, 0.38])
share_full = yearly.loc[:LAST_FULL, "share"]   # the partial year's share is not comparable (shown as a count only)
# the external-partner share only in the years where it may be shown (small groups): gaps elsewhere, and no value
# in the figure data
ext_shown = ext_year_shown.reindex(share_full.index, fill_value=False).astype(bool)
share_ext_shown = [float(v) if ok else None for v, ok in zip(yearly.loc[:LAST_FULL, "share_ext"], ext_shown)]
fig.add_bar(x=yearly.index, y=yearly["intl"], name="International", marker_color=INTL_COLOR,
            customdata=yearly[["papers", "share"]].to_numpy(),
            hovertemplate=["%{y:,} of %{customdata[0]:,} (partial year)" if y >= PARTIAL
                           else "%{y:,} of %{customdata[0]:,} (%{customdata[1]:.1%})" for y in yearly.index],
            row=1, col=1)
fig.add_bar(x=yearly.index, y=yearly["greece"], name="Greece only", marker_color=dp.OTHER_COLOR,
            hovertemplate="%{y:,}", row=1, col=1)
fig.add_scatter(x=share_full.index, y=share_full, mode="lines", name="International share",
                line=dict(color=INTL_COLOR, width=2), showlegend=False,
                hovertemplate=["%{y:.1%}" if ok else "%{y:.1%}; external-partner share not shown (small groups)"
                               for ok in ext_shown],
                row=2, col=1)
fig.add_scatter(x=share_full.index, y=share_ext_shown, mode="lines+markers",
                name="External-partner share", line=dict(color=INTL_COLOR, width=2, dash="dash"),
                marker=dict(color=INTL_COLOR, size=4), showlegend=False, hovertemplate="%{y:.1%}", row=2, col=1)
fig.update_layout(barmode="stack", hovermode="x unified", height=480, margin=dict(t=24))
fig.update_yaxes(title_text="Publications", row=1, col=1)
fig.update_yaxes(title_text="International share", tickformat=".0%", rangemode="tozero", row=2, col=1)
dp.style_year_axis(fig)
dp.style_partial_year(fig, PARTIAL, as_of=AS_OF)
fig.show()

## Output and international co-authorship

:::{figure} #cell-overview-output
:label: fig-overview-output
Publications per year, {eval}`str(CHART_FROM)`–{eval}`str(PARTIAL)`. Top: papers with at least one
affiliation outside Greece (international) and papers without one (Greece only). Bottom: share of
international papers (solid) and the external-partner share, counting only affiliations of co-authors
who are not SECE authors (dashed), full years only. The dashed line has gaps in the years where its value,
alone or with other values on this site (such as the shares by period on [Collaboration](collaboration.ipynb)),
would give the number of papers that are international only through the own second affiliations of fewer than
{eval}`str(MIN_HOLDERS)` SECE authors (the same years as in that page's chart; see the note below).
{eval}`str(PARTIAL)` is partial (records to
{eval}`AS_OF_TEXT`). The {eval}`dp.fmt_int(n_before_chart)` records of
{eval}`str(FIRST_YEAR)`–{eval}`str(CHART_FROM - 1)`, at most {eval}`dp.fmt_int(max_before_chart)` a
year, are not shown.
:::

The School published {eval}`last_year_text`, and
{eval}`dp.fmt_pct(intl_year[LAST_FULL])` of them were international, {eval}`intl_last_text`; the
external-partner share, {eval}`dp.fmt_pct(intl_year_ext[LAST_FULL])`, was {eval}`ext_last_text`
([](#fig-overview-output)).

## Highlights

- **[Research output](output.ipynb).** The {eval}`dp.fmt_int(recent_n)` publications of
  {eval}`P_RECENT` are {eval}`growth_text` than in {eval}`P_PREV`, and teams have
  grown: a publication had {eval}`dp.fmt_num(team[P_BASE], 1)` authors on average in {eval}`P_BASE`
  and {eval}`dp.fmt_num(team[P_RECENT], 1)` in {eval}`P_RECENT`.
- **[Citation impact](impact.ipynb).** Citations are concentrated: the most-cited
  {eval}`dp.fmt_pct(TOP_FRAC, 0)` of the papers hold {eval}`dp.fmt_pct(top10_share)` of all citations (a
  median {eval}`dp.fmt_pct(cohort_top10)` within each publication year from {eval}`str(NCS_FROM)` to
  {eval}`str(NCS_TO)`). Compared with the School's papers of the same year and type
  ({eval}`str(NCS_FROM)`–{eval}`str(NCS_TO)`), internationally co-authored papers are cited
  {eval}`vs_average(ncs_intl)` the average, mainly those led by partners (a mean normalised score of
  {eval}`dp.fmt_num(ncs_intl_partner, 2)` when SECE holds neither the first nor the last author position,
  {eval}`dp.fmt_num(ncs_intl_led, 2)` when it holds one of them), and papers that acknowledge funding
  {eval}`vs_average(ncs_funded)`. These are associations, not measured effects.
- **[Venues & publishers](venues.ipynb).** IEEE published {eval}`dp.fmt_pct(share_ieee)` of all
  records ({eval}`dp.fmt_int(n_ieee)`) and {eval}`dp.fmt_pct(ieee_recent)` of those of
  {eval}`P_RECENT`. MDPI grew from {eval}`dp.fmt_pct(mdpi_prev)` of the output of {eval}`P_PREV` to
  {eval}`dp.fmt_pct(mdpi_recent)` in {eval}`P_RECENT`.
- **[Research topics](topics.ipynb).** Of the {eval}`str(K_TOPICS)` research topics found in the
  titles, {eval}`topic_gain` gained the most share, from {eval}`dp.fmt_pct(gain_from)` of the titles in
  {eval}`P_DECADE` to {eval}`dp.fmt_pct(gain_to)` in {eval}`P_RECENT`. {eval}`losses_text` lost the
  most.
- **[Collaboration](collaboration.ipynb).** International co-authorship {eval}`intl_fold`, from
  {eval}`dp.fmt_pct(intl[P_BASE])` of the papers in {eval}`P_BASE` to {eval}`dp.fmt_pct(intl[P_RECENT])`
  in {eval}`P_RECENT`, or from {eval}`dp.fmt_pct(intl_ext[P_BASE])` to
  {eval}`dp.fmt_pct(intl_ext[P_RECENT])` counting only affiliations of co-authors who are not SECE
  authors: {eval}`dp.fmt_int(n_own_intl_recent)` of the {eval}`dp.fmt_int(n_intl_recent)` international
  papers of {eval}`P_RECENT` are international only through SECE authors' own second affiliations abroad.
  Papers with AUTH as the only institution fell from {eval}`dp.fmt_pct(single[P_BASE])` to
  {eval}`dp.fmt_pct(single[P_RECENT])`.
- **[Global reach](geography.ipynb).** {eval}`dp.fmt_int(n_intl)` of the {eval}`dp.fmt_int(n_pubs)`
  papers ({eval}`dp.fmt_pct(share_intl_all)`) list at least one affiliation abroad; the most frequent partner
  country is {eval}`country_1` ({eval}`dp.fmt_int(country_1_n)` papers), ahead of {eval}`country_2`
  ({eval}`dp.fmt_int(country_2_n)`). The reach has widened from {eval}`dp.fmt_num(countries_early, 1)`
  partner countries a year in {eval}`P_BASE` to {eval}`dp.fmt_num(countries_recent, 1)` in
  {eval}`P_RECENT`.
- **[Funding & open access](funding.ipynb).** The share of publications that acknowledge funding rose
  from {eval}`dp.fmt_pct(funded[P_DECADE])` in {eval}`P_DECADE` to {eval}`dp.fmt_pct(funded[P_RECENT])` in
  {eval}`P_RECENT` (part of the rise may reflect more complete indexing of acknowledgements by Scopus),
  and the open-access share from {eval}`dp.fmt_pct(oa[P_DECADE])` to
  {eval}`dp.fmt_pct(oa[P_RECENT])`. In {eval}`P_RECENT`, journal publications were open access far
  more often ({eval}`dp.fmt_pct(oa_kind['journal'])`) than conference papers
  ({eval}`dp.fmt_pct(oa_kind['conference'])`).
- **[Research community](people.ipynb).** An average of {eval}`dp.fmt_int(active_recent)` author
  profiles a year published with an SECE affiliation in {eval}`P_RECENT`,
  {eval}`dp.fmt_num(active_growth, 1)` times as many as in {eval}`P_BASE`
  ({eval}`dp.fmt_int(active_early)`), while the number of papers with an SECE author grew
  {eval}`dp.fmt_num(papers_growth, 1)`-fold.

:::{note} About this site
This is an unofficial analysis: it is not a publication of AUTH or of the School, and its figures
are not official statistics. Every page uses the same Scopus export of {eval}`AS_OF_TEXT`, with the
citation counts of that date (self-citations included). Only documents indexed in Scopus are counted,
each publication counts once however many co-authors it has (unless a page says otherwise), and the
pages show School-level aggregates only: no author names, identifiers or individual metrics.
Collaboration figures count every affiliation on a paper, SECE authors' own second affiliations
included, unless they are said to count only affiliations of co-authors who are not SECE authors.
Small groups are not shown: a number of papers that link to something (any country abroad, a country,
a region, an institution, a sector or a collaboration type) only through SECE authors' own second
affiliations, in any year or period, is given only when it is zero or at least
{eval}`str(MIN_HOLDERS)` SECE authors hold these affiliations, since with fewer it would in practice be
one or two people's paper count. This also covers numbers that follow from others, on one page or
across pages, such as the difference between the two bases or a period minus its years, so values that
would reveal such a number are left out (the gaps in the dashed line of [](#fig-overview-output)).
Partner institutions come from linking the affiliations to the
Research Organization Registry (ROR, {eval}`ROR_TEXT`), which succeeded for
{eval}`dp.fmt_pct(coverage_ext)` of the affiliation occurrences outside AUTH, on which the partner counts
rest ({eval}`dp.fmt_pct(coverage)` of all occurrences). [Methodology & data](methodology.ipynb)
describes the sources, the definitions, the validation of this matching and the limitations.
:::